# Credit Risk Simulator

In this simulator, I initially build a simple case using a randomly generated population with three randomly selected variables (age, car value, and province) to estimate the Probability of Default (PD). As will be shown later, a fully random dataset does not realistically represent a real-world portfolio, since there is no relationship between the explanatory variables and the target. Therefore, a second dataset is created with more realistic characteristics to better mimic the behaviour observed in actual credit portfolios.

In [1]:
import numpy as np
import pandas as pd

from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
# ---------------------------------------------------------
# TRAINING DATA
# ---------------------------------------------------------

np.random.seed(42)
n = 10000

df = pd.DataFrame({
    "edad": np.random.randint(18, 70, n),
    "precio_coche": np.random.randint(3000, 50000, n),
    "provincia": np.random.choice(
        ["Madrid", "Barcelona", "Sevilla"], n
    ),
    "target": np.random.choice(
        [0, 1],
        n,
        p=[0.8, 0.2]
    )
})

# ---------------------------------------------------------
# BINNING AND WEIGHT OF EVIDENCE (WOE)
# ---------------------------------------------------------

df["bin_edad"], edges_edad = pd.qcut(
    df["edad"],
    q=3,
    retbins=True,
    duplicates="drop"
)

df["bin_coche"], edges_coche = pd.qcut(
    df["precio_coche"],
    q=3,
    retbins=True,
    duplicates="drop"
)

df["bin_provincia"] = df["provincia"]


def calc_woe(df, var):

    t = (
        df.groupby(var, observed=False)["target"]
        .agg(bads="sum", total="count")
    )

    t["goods"] = t["total"] - t["bads"]

    # Suavizado para evitar log(0)
    t["goods"] += 0.5
    t["bads"] += 0.5

    pct_goods = t["goods"] / t["goods"].sum()  #pct = percentage
    pct_bads = t["bads"] / t["bads"].sum()

    return np.log(pct_goods / pct_bads).to_dict()  # Maps only work with dictionaries


map_edad = calc_woe(df, "bin_edad")
map_coche = calc_woe(df, "bin_coche")
map_prov = calc_woe(df, "bin_provincia")

df["woe_edad"] = df["bin_edad"].map(map_edad).astype(float)
df["woe_coche"] = df["bin_coche"].map(map_coche).astype(float)
df["woe_prov"] = df["bin_provincia"].map(map_prov).astype(float)

X = df[["woe_edad", "woe_coche", "woe_prov"]]
y = df["target"]

# ---------------------------------------------------------
# LOGISTIC REGRESSION
# ---------------------------------------------------------

model = LogisticRegression()

model.fit(X, y)

df["pd"] = model.predict_proba(X)[:, 1]

# ---------------------------------------------------------
# DISCRIMINATION
# ---------------------------------------------------------

auc = roc_auc_score(y, df["pd"])
gini = 2 * auc - 1

df_sort = df.sort_values("pd", ascending=False)

cum_bads = (
    (df_sort["target"] == 1).cumsum()
    / y.sum()
)

cum_goods = (
    (df_sort["target"] == 0).cumsum()
    / (len(y) - y.sum())
)

ks = np.max(np.abs(cum_bads - cum_goods))

print("=== DISCRIMINACIÓN ===")
print(f"GINI: {gini:.3f}")
print(f"KS:   {ks*100:.2f}%\n")

# ---------------------------------------------------------
# CALIBRATION
# ---------------------------------------------------------

p_binom = stats.binomtest(
    y.sum(),
    n=len(y),
    p=df["pd"].mean()
).pvalue

df["decil"] = pd.qcut(
    df["pd"],
    q=10,
    duplicates="drop"
)

hl = (
    df.groupby("decil", observed=False)
    .agg(
        obs=("target", "sum"), # impagos observados
        exp=("pd", "sum"),     # suma de probs de impago experimentales
        n=("target", "count")
    )
)

hl_stat = np.sum(
    (hl["obs"] - hl["exp"]) ** 2
    / (hl["exp"] * (1 - hl["exp"] / hl["n"]))  # Es un chi cuadrado -> sum (Obs - Exp)^2 / (varianza)
)

p_hl = 1 - stats.chi2.cdf( # equivale a P(chi^2 > hl_stat)
    hl_stat,
    df=len(hl) - 2
)

print("=== CALIBRACIÓN ===")
print(f"P-value Binomial: {p_binom:.4f}")
print(f"P-value Hosmer-Lemeshow: {p_hl:.4f}\n")

# ---------------------------------------------------------
# NEW SAMPLE
# ---------------------------------------------------------
N = 500
df_new = pd.DataFrame({
    "edad": np.random.randint(20, 60, N),
    "precio_coche": np.random.randint(5000, 45000, N),
    "provincia": np.random.choice(
        ["Madrid", "Barcelona", "Sevilla"],
        N
    ),
    "target": np.random.choice(
        [0, 1],
        N,
        p=[0.75, 0.25]
    )
})

df_new["bin_edad"] = pd.cut(
    df_new["edad"],
    bins=edges_edad,
    include_lowest=True
)

df_new["bin_coche"] = pd.cut(
    df_new["precio_coche"],
    bins=edges_coche,
    include_lowest=True
)

df_new["bin_provincia"] = df_new["provincia"]

# ---------------------------------------------------------
# WOE
# ---------------------------------------------------------

X_new = pd.DataFrame({
    "woe_edad": df_new["bin_edad"].map(map_edad),
    "woe_coche": df_new["bin_coche"].map(map_coche),
    "woe_prov": df_new["bin_provincia"].map(map_prov)
})

# ---------------------------------------------------------
# PD NEW SAMPLE
# ---------------------------------------------------------

df_new["pd"] = model.predict_proba(X_new)[:, 1]

# ---------------------------------------------------------
# PSI
# ---------------------------------------------------------

q_bins = pd.qcut(
    df["pd"],
    q=10,
    retbins=True,
    duplicates="drop"
)[1]

q_bins[0] = -np.inf
q_bins[-1] = np.inf

train_dist = (
    pd.cut(df["pd"], bins=q_bins)
    .value_counts(normalize=True, sort=False)
)

new_dist = (
    pd.cut(df_new["pd"], bins=q_bins)
    .value_counts(normalize=True, sort=False)
)

eps = 1e-6

psi = np.sum(
    ((new_dist + eps) - (train_dist + eps))
    * np.log((new_dist + eps) / (train_dist + eps))
)

print("=== ESTABILIDAD TEMPORAL ===")
print(f"PSI: {psi:.4f}")

=== DISCRIMINACIÓN ===
GINI: 0.013
KS:   2.14%

=== CALIBRACIÓN ===
P-value Binomial: 1.0000
P-value Hosmer-Lemeshow: 0.1730

=== ESTABILIDAD TEMPORAL ===
PSI: 0.0349


- GINI = 0.013 is essentially zero. However, this is exactly what we would expect from this simulation. All variables, including the target, are generated randomly, meaning that there is no underlying relationship between the predictors and the target. As a result, the model is unable to learn any meaningful patterns because there are none to learn.
- KS = 2% confirms the same conclusion. There is virtually no predictive signal in the data, and therefore the model cannot effectively distinguish between good and bad borrowers.
- The Binomial Test p-value is extremely high because the model is both fitted and evaluated on the same dataset.
- PSI = 0.035 indicates that the distribution of the new population is very similar to that of the training population.

# Improving the Model
- The first logical step is to make the input variables more realistic rather than completely random.

In [2]:
n = 10000
df = pd.DataFrame({
    # Más gente entre 30 y 55 años que en los extremos
    "edad": np.clip(
        np.random.normal(42, 12, n).astype(int),
        18,
        75
    ),

    # Distribución muy sesgada: muchos coches baratos, pocos caros
    "precio_coche": np.clip(
        np.random.lognormal(
            mean=np.log(15000),
            sigma=0.6,
            size=n
        ).astype(int),
        2000,
        100000
    ),

    # Provincias con pesos más realistas
    "provincia": np.random.choice(
        ["Madrid", "Barcelona", "Sevilla"],
        size=n,
        p=[0.5, 0.3, 0.2]
    )
})

logit = (
    1.5
    - 0.035 * df["edad"]                # más edad -> menos riesgo
    - 0.00004 * df["precio_coche"]      # coche más caro -> menos riesgo
    + np.where(df["provincia"] == "Sevilla", 0.6, 0)  # Más riesgo en Sevilla y Barcelona
    + np.where(df["provincia"] == "Barcelona", 0.2, 0)
    + np.random.normal(0, 0.5, n)       # ruido
)

pd_real = 1 / (1 + np.exp(-logit))

df["target"] = np.random.binomial(
    n=1,
    p=pd_real
)

- We then recalculate the entire modelling process using the improved dataset.

In [3]:
df["bin_edad"], edges_edad = pd.qcut(
    df["edad"],
    q=3,
    retbins=True,
    duplicates="drop"
)

df["bin_coche"], edges_coche = pd.qcut(
    df["precio_coche"],
    q=3,
    retbins=True,
    duplicates="drop"
)

df["bin_provincia"] = df["provincia"]


def calc_woe(df, var):

    t = (
        df.groupby(var, observed=False)["target"]
        .agg(bads="sum", total="count")
    )

    t["goods"] = t["total"] - t["bads"]

    # Suavizado para evitar log(0)
    t["goods"] += 0.5
    t["bads"] += 0.5

    pct_goods = t["goods"] / t["goods"].sum()  #pct = percentage
    pct_bads = t["bads"] / t["bads"].sum()

    return np.log(pct_goods / pct_bads).to_dict()  # Maps only work with dictionaries


map_edad = calc_woe(df, "bin_edad")
map_coche = calc_woe(df, "bin_coche")
map_prov = calc_woe(df, "bin_provincia")

df["woe_edad"] = df["bin_edad"].map(map_edad).astype(float)
df["woe_coche"] = df["bin_coche"].map(map_coche).astype(float)
df["woe_prov"] = df["bin_provincia"].map(map_prov).astype(float)

X = df[["woe_edad", "woe_coche", "woe_prov"]]
y = df["target"]

# ---------------------------------------------------------
# LOGISTIC REGRESSION
# ---------------------------------------------------------

model = LogisticRegression()

model.fit(X, y)

df["pd"] = model.predict_proba(X)[:, 1]

# ---------------------------------------------------------
# DISCRIMINATION
# ---------------------------------------------------------

auc = roc_auc_score(y, df["pd"])
gini = 2 * auc - 1

df_sort = df.sort_values("pd", ascending=False)

cum_bads = (
    (df_sort["target"] == 1).cumsum()
    / y.sum()
)

cum_goods = (
    (df_sort["target"] == 0).cumsum()
    / (len(y) - y.sum())
)

ks = np.max(np.abs(cum_bads - cum_goods))

print("=== DISCRIMINACIÓN ===")
print(f"GINI: {gini:.3f}")
print(f"KS:   {ks*100:.2f}%\n")

# ---------------------------------------------------------
# CALIBRATION
# ---------------------------------------------------------

p_binom = stats.binomtest(
    y.sum(),
    n=len(y),
    p=df["pd"].mean()
).pvalue

df["decil"] = pd.qcut(
    df["pd"],
    q=10,
    duplicates="drop"
)

hl = (
    df.groupby("decil", observed=False)
    .agg(
        obs=("target", "sum"), # impagos observados
        exp=("pd", "sum"),     # suma de probs de impago experimentales
        n=("target", "count")
    )
)

hl_stat = np.sum(
    (hl["obs"] - hl["exp"]) ** 2
    / (hl["exp"] * (1 - hl["exp"] / hl["n"]))  # Es un chi cuadrado -> sum (Obs - Exp)^2 / (varianza)
)

p_hl = 1 - stats.chi2.cdf( # equivale a P(chi^2 > hl_stat)
    hl_stat,
    df=len(hl) - 2
)

print("=== CALIBRACIÓN ===")
print(f"P-value Binomial: {p_binom:.4f}")
print(f"P-value Hosmer-Lemeshow: {p_hl:.4f}\n")

# ---------------------------------------------------------
# NEW SAMPLE
# ---------------------------------------------------------
N = 500
df_new = pd.DataFrame({
    "edad": np.random.randint(20, 60, N),
    "precio_coche": np.random.randint(5000, 45000, N),
    "provincia": np.random.choice(
        ["Madrid", "Barcelona", "Sevilla"],
        N
    ),
    "target": np.random.choice(
        [0, 1],
        N,
        p=[0.75, 0.25]
    )
})

df_new["bin_edad"] = pd.cut(
    df_new["edad"],
    bins=edges_edad,
    include_lowest=True
)

df_new["bin_coche"] = pd.cut(
    df_new["precio_coche"],
    bins=edges_coche,
    include_lowest=True
)

df_new["bin_provincia"] = df_new["provincia"]

# ---------------------------------------------------------
# WOE
# ---------------------------------------------------------

X_new = pd.DataFrame({
    "woe_edad": df_new["bin_edad"].map(map_edad),
    "woe_coche": df_new["bin_coche"].map(map_coche),
    "woe_prov": df_new["bin_provincia"].map(map_prov)
})

# ---------------------------------------------------------
# PD NEW SAMPLE
# ---------------------------------------------------------

df_new["pd"] = model.predict_proba(X_new)[:, 1]

# ---------------------------------------------------------
# PSI
# ---------------------------------------------------------

q_bins = pd.qcut(
    df["pd"],
    q=10,
    retbins=True,
    duplicates="drop"
)[1]

q_bins[0] = -np.inf
q_bins[-1] = np.inf

train_dist = (
    pd.cut(df["pd"], bins=q_bins)
    .value_counts(normalize=True, sort=False)
)

new_dist = (
    pd.cut(df_new["pd"], bins=q_bins)
    .value_counts(normalize=True, sort=False)
)

eps = 1e-6

psi = np.sum(
    ((new_dist + eps) - (train_dist + eps))
    * np.log((new_dist + eps) / (train_dist + eps))
)

print("=== ESTABILIDAD TEMPORAL ===")
print(f"PSI: {psi:.4f}")

=== DISCRIMINACIÓN ===
GINI: 0.269
KS:   20.09%

=== CALIBRACIÓN ===
P-value Binomial: 0.9918
P-value Hosmer-Lemeshow: 0.5415

=== ESTABILIDAD TEMPORAL ===
PSI: 0.1734


## Results with the Updated Dataset

- GINI = 0.27 indicates the presence of predictive power. For a simple scorecard based on only a few variables, this is a plausible result.
- KS = 20% is consistent with the observed Gini coefficient. Although the model is far from perfect, it demonstrates a reasonable ability to rank borrowers according to risk.
- Calibration: The calibration results are not surprising, as the model is still being evaluated on the same dataset used for training.
- PSI = 0.16 is perhaps the most interesting metric. It shows that the new population differs from the original one. A newly generated sample was intentionally introduced to assess the impact on the PSI, and the metric successfully captures and quantifies the population shift, indicating that the new data exhibit different characteristics and behavioural patterns.